# NB08 · Retention Strategy & Recommendations
**Purpose:** Data-backed recommendations to reduce churn, segmented by band and churn reason, with model deployment guidance.

In [0]:
%run ./NB01_Config_and_Setup

In [0]:
billings = load_table(BILLINGS_TABLE, BILLINGS_PATH)
billings = parse_dates(billings, ['Prospect_Renewal_Date','Closed_Date'])
billings['days_to_close'] = (billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days
conditions = [
    (billings['Prospect_Outcome']=='Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome']=='Won', billings['Prospect_Outcome']=='Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned','Churned','Pre_Churned','Won','Open'], default='Churned')
model_df = billings[billings['Churn_Label'].isin(['Won','Churned'])].copy()
model_df['Target']       = (model_df['Churn_Label']=='Churned').astype(int)
model_df['Renewal_Year'] = model_df['Prospect_Renewal_Date'].dt.year
mdf = model_df[model_df['Renewal_Year'].isin(TARGET_YEARS)].copy()
print(f"[NB08] {len(mdf):,} records loaded")


## 1 · Revenue Impact of Churn

In [0]:
section("Revenue Impact Analysis")
for c in ['Last_Years_Price','Starting_Net','Total_Net_Paid']:
    if c in mdf.columns:
        mdf[c] = pd.to_numeric(mdf[c], errors='coerce')

churned = mdf[mdf['Target']==1]

if 'Last_Years_Price' in mdf.columns:
    avg_price_churned = churned['Last_Years_Price'].median()
    total_churned     = len(churned)
    annual_rev_lost   = avg_price_churned * total_churned

    print(f"Median renewal price (churned accounts) : £{avg_price_churned:,.0f}")
    print(f"Total churned accounts (2023–2025)       : {total_churned:,}")
    print(f"Estimated total revenue lost             : £{annual_rev_lost:,.0f}")
    print()

    band_rev = churned.groupby('Band').agg(
        Churned=('Target','count'),
        Med_Price=('Last_Years_Price','median')
    ).reset_index()
    band_rev['Est_Rev_Lost'] = (band_rev['Churned'] * band_rev['Med_Price']).round(0)
    band_rev = band_rev.sort_values('Est_Rev_Lost', ascending=False)
    band_rev['10pct_Saving'] = (band_rev['Est_Rev_Lost'] * 0.10).round(0)
    display_df(band_rev, "Revenue Loss and 10% Retention Saving by Band")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(band_rev['Band'], band_rev['Est_Rev_Lost']/1000,
            color=[rate_color(r/band_rev['Est_Rev_Lost'].max()*15) for r in band_rev['Est_Rev_Lost']],
            edgecolor='#0a0e1a')
    ax.set_xlabel("Estimated Revenue Lost (£ thousands)")
    ax.set_title("Estimated Revenue Lost by Band", fontsize=13)
    for i,(_, row) in enumerate(band_rev.iterrows()):
        ax.text(row['Est_Rev_Lost']/1000+1, i,
                f"£{row['Est_Rev_Lost']/1000:.0f}K", va='center', fontsize=9)
    plt.tight_layout(); plt.show()


## 2 · Recommendations — Band-Specific Strategies

In [0]:
section("Band-Specific Retention Strategies")
recs = {
    'Band A & B — Entry Tier (Churn: 16.8%, 13.8%)': [
        'ROOT CAUSE: Price-to-value disconnect. These are smaller contractors paying relatively high fees compared to their anchor client base.',
        'ACTION 1: Introduce a tiered value demonstration — send monthly ROI summary emails showing how many client opportunities their accreditation enabled.',
        'ACTION 2: Implement an early-warning auto-flag at 90 days pre-renewal for all Band A/B accounts with fewer than 2 active anchor clients.',
        'ACTION 3: Equip renewal agents with a structured value script specific to Band A/B, addressing the "not worth it" objection with concrete comparisons.',
        'ACTION 4: Pilot a loyalty pricing model — first-renewal discount of 10-15% for Band A members with tenure < 2 years, graduated back up over 3 years.',
    ],
    'Band C1 & C2 — Mid Tier (Churn: 8.7%, 6.5%)': [
        'ROOT CAUSE: Accreditation delays and platform friction. Customers at this level are investing more time in the process and expect smoother delivery.',
        'ACTION 1: Proactively contact accounts where questionnaire completion rate falls below 60% at 60 days before renewal — offer a dedicated support call.',
        'ACTION 2: Track platform-issue flags from CC calls. Any customer with 2+ platform complaints in the renewal window triggers a Service Recovery protocol.',
        'ACTION 3: Introduce "progress towards accreditation" nudge emails — showing percentage complete reduces the perceived complexity that drives churn at this band.',
    ],
    'Band D to Group — Higher Tiers (Churn: <6%)': [
        'ROOT CAUSE: Lower churn driven by deeper embedding — more anchor clients, more to lose by switching.',
        'ACTION 1: Maintain the anchor relationship. Any account that drops below its usual anchor count should trigger a relationship manager review.',
        'ACTION 2: For accounts with competitors mentioned in any channel, escalate to a senior retention manager before the renewal call.',
        'ACTION 3: Offer multi-year renewal pricing for accounts with tenure > 5 years — locks in revenue and reduces annual renewal friction.',
    ],
}
for band_group, actions in recs.items():
    print(f"\n{'─'*65}")
    print(f"  {band_group}")
    print(f"{'─'*65}")
    for a in actions:
        print(f"    • {a}")


## 3 · Recommendations — Churn Reason Playbook

In [0]:
section("Churn Reason Playbook")
playbook = {
    'Not Value for Money (1,782 accounts)': [
        'TIMING: Intervene at 120 days pre-renewal, not at 30 days — by then the decision is made.',
        'SCRIPT: Lead with anchor client count and industry benchmarks, not product features.',
        'TOOL: Build a personalised "Value Statement" PDF auto-generated from the customer\'s own data.',
    ],
    'Do Not Work for Client (1,671 accounts)': [
        'DETECTION: Flag accounts where the anchor client\'s industry code appears in a distressed sector watch list.',
        'ACTION: Offer a 60-day suspension option instead of full cancellation — reduces loss-to-cancel conversion.',
        'PRODUCT: Develop a "dormant membership" lower-cost tier for contractors between major clients.',
    ],
    'No Longer Trading (1,225 accounts)': [
        'PREVENTION: Limited — but early detection via Companies House API integration could flag dissolution filings.',
        'ACTION: For sole traders showing reducing invoice volumes (if data available), trigger a support outreach before closure.',
    ],
    'Refused to Discuss / Non-Responsive (1,723 combined)': [
        'ROOT CAUSE: Customers had already decided. The issue is that no engagement happened earlier in the cycle.',
        'ACTION: Implement a 3-touch nurture sequence starting 90 days before renewal for all accounts with no email open in the prior 60 days.',
        'CHANNEL: Add SMS nudge as a third channel for non-responsive accounts — email open rates alone are insufficient.',
    ],
    'Competitor Accreditation (769 accounts)': [
        'INTELLIGENCE: Log all competitor names mentioned across all channels. Build a competitor comparison card for agents.',
        'RESPONSE: Train agents to acknowledge the competitor by name, highlight specific differentiators, and offer a comparison call.',
        'PRODUCT: Monitor SSIP approval and DTS scheme adoption — if a competitor achieves these, the value proposition narrows.',
    ],
    'Not Affordable (578 accounts)': [
        'SEGMENT: Cross-reference with CC call data — accounts flagging financial hardship should be flagged 180 days pre-renewal.',
        'OFFER: Create a payment-plan option (3-month spread) for accounts at risk. Even partial retention beats full churn.',
        'ESCALATION: Give senior agents discretionary authority to offer a 15% hardship reduction without manager approval.',
    ],
}
for reason, actions in playbook.items():
    print(f"\n{'─'*65}")
    print(f"  {reason}")
    print(f"{'─'*65}")
    for a in actions:
        print(f"    • {a}")


## 4 · Model Deployment Plan

In [0]:
section("Model Deployment — Recommended Architecture")
deployment_plan = """
PHASE 1 — SCORING (Week 1-2)
  ├─ Schedule daily batch job: score all accounts due for renewal in next 90 days
  ├─ Output: Co_Ref + Churn_Probability_Score + Risk_Tier (High/Medium/Low)
  └─ Load scores to CRM dashboard

PHASE 2 — ROUTING (Week 3-4)
  ├─ High Risk  (score >= 0.65) → Senior retention agent + value script + discount authority
  ├─ Medium Risk (0.35–0.65)   → Standard renewal agent + value script
  └─ Low Risk   (< 0.35)       → Automated email sequence only

PHASE 3 — A/B TEST (Month 2-3)
  ├─ Split Band B into model-routed (treatment) vs. business-as-usual (control)
  ├─ Match on: renewal month, tenure, last year's price
  └─ Measure: churn rate, revenue retained, agent time per account

PHASE 4 — FEEDBACK LOOP (Ongoing)
  ├─ Every quarter: retrain model on latest 12 months of data
  ├─ Every 6 months: re-run NB06 feature selection — signal may shift
  └─ Monitor: model AUC on live cohort (flag if drops below 0.85)
"""
print(deployment_plan)


## 5 · Summary KPI Targets

In [0]:
section("Recommended KPI Targets — 12 Month Horizon")
targets = {
    'Overall Churn Rate'         : ('Current: 7.28%',  'Target: < 5.5%',  'Gap: -1.78pp'),
    'Band A Churn Rate'          : ('Current: 16.79%', 'Target: < 12%',   'Gap: -4.79pp'),
    'Band B Churn Rate'          : ('Current: 13.76%', 'Target: < 10%',   'Gap: -3.76pp'),
    'Non-Responsive Rate'        : ('Current: ~16% of churned', 'Target: < 10%', 'Via 90-day nurture'),
    'Value Perception Churn'     : ('Current: #1 reason', 'Target: Drop to #3 or lower', 'Via ROI comms'),
    'Model Coverage'             : ('Current: 0%', 'Target: 100% of renewal cohort scored daily','Deploy in Phase 1'),
    'Avg Days to First Intervention': ('Current: ~30 days pre-renewal', 'Target: 90 days','Via early scoring'),
}
target_df = pd.DataFrame(targets, index=['Current','Target','Action/Gap']).T.reset_index()
target_df.columns = ['KPI','Current','Target','Action/Gap']
display_df(target_df, "12-Month Retention KPI Targets")


In [0]:
section("Fact-Backed Pitch Summary")
summary = """
WHAT THE DATA PROVES (from analysis across 110,878 accounts, 2023-2025):

  1. 2023 was the crisis year: churn hit 13.89% in July — nearly 1 in 7 members left.
     → Training on this period gives the model the most adversarial signal available.

  2. Band A and Band B are structurally fragile: 16.8% and 13.8% churn rates, driven
     primarily by price-value disconnect and competitor alternatives.
     → These two bands account for 46% of all churned accounts.

  3. The problem is improving but plateauing: 2024 dropped to 7.98%, 2025 to 7.28%.
     Without intervention, it is unlikely to fall below 6-7% on its own.
     → The model creates the headroom to push past that plateau.

  4. Silent churn is the hardest problem: 1,723 accounts churned without any meaningful
     engagement (refused to discuss or non-responsive). By the time the agent calls, it is too late.
     → The model flags these accounts 90 days earlier — before disengagement hardens.

  5. Every 1% reduction in churn rate saves approximately £500K-£1.5M in annual recurring
     revenue (based on median renewal prices). A 10% lift in Band B retention alone saves
     over 400 accounts per year.
     → The model pays for itself in the first quarter if deployed correctly.
"""
print(summary)
